# Test Loading a recorded video file

In [ ]:
# Setting a path
from pathlib import Path

video_path = Path("../data/IMG_4456.MOV")
print(video_path)

In [ ]:

# Loading instance
from baseball_motion_analysis.video import FrameSamplingOptions, MediaInputService

service = MediaInputService()

sequence = service.load_video_file(
    Path(video_path),
    sampling=FrameSamplingOptions(sample_every_n_frames=5),
)

In [ ]:
import cv2
import matplotlib.pyplot as plt

frame = sequence.frames[0]

rgb = cv2.cvtColor(frame.image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(rgb)
plt.axis("off")
plt.title(f"frame={frame.frame_index}, time={frame.timestamp_seconds:.3f}s")
plt.show()

In [ ]:
print(sequence.source_type)
print(sequence.metadata.width, sequence.metadata.height)
print(sequence.metadata.fps)
print(sequence.metadata.total_frame_count)
print(sequence.sampled_frame_count)
print(sequence.frames[0].frame_index)
print(sequence.frames[0].timestamp_seconds)

# Pose estimation

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_styles, drawing_utils


def mp_image_to_bgr(image):
    image_array = image.numpy_view()
    if image_array.ndim == 2:
        return cv2.cvtColor(image_array, cv2.COLOR_GRAY2BGR)
    if image_array.shape[2] == 4:
        return cv2.cvtColor(image_array, cv2.COLOR_RGBA2BGR)
    return cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR)


def draw_landmarks_on_image(bgr_image, detection_result):
    pose_landmarks_list = detection_result.pose_landmarks
    annotated_image = np.ascontiguousarray(bgr_image.copy())

    pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
    pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

    for pose_landmarks in pose_landmarks_list:
        drawing_utils.draw_landmarks(
            image=annotated_image,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=pose_landmark_style,
            connection_drawing_spec=pose_connection_style,
        )

    return annotated_image


In [ ]:
# STEP 1: Import the necessary modules.
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# STEP 2: Create a PoseLandmarker object.
base_options = python.BaseOptions(model_asset_path="../models/pose_landmarker_full.task")
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True,
)
detector = vision.PoseLandmarker.create_from_options(options)

# STEP 3: Load the input image.
image = mp.Image.create_from_file("../data/pose.jpg")

# STEP 4: Detect pose landmarks from the input image.
detection_result = detector.detect(image)

# STEP 5: Process the detection result. In this case, visualize it.
image_bgr = mp_image_to_bgr(image)
annotated_image = draw_landmarks_on_image(image_bgr, detection_result)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()
